# Visualizations

Demos for the geometric VVAE pipeline. Run top to bottom.

## Setup

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 3
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from hexai.vvae import lift_window, render_view
from hexai.vvae.data import TartanGroundWindowDataset, find_trajectory_dirs

house_root = Path("../../../../data/tartanground/House/Data_omni")
bounds = torch.tensor([[-2.0, -1.5, 0.0], [2.0, 1.5, 4.0]])  # metres, frame 0's camera
grid_res, render_hw = (32, 24, 32), (160, 160)

posed = TartanGroundWindowDataset(
    trajectory_paths=find_trajectory_dirs(house_root),
    num_frames=5, image_hw=(160, 160), include_pose=True,
)
frames, poses = posed[6]
volume = lift_window(frames, poses, bounds, grid_res, fovy_deg=90.0)
print(f"{len(posed)} windows | volume {tuple(volume.shape)}, "
      f"{(volume[3] > 1e-6).float().mean() * 100:.1f}% occupied")

## What a window holds

Five frames of rgb over colormapped depth.

In [ ]:
fig, axes = plt.subplots(2, frames.shape[1], figsize=(13, 5.4))
for t in range(frames.shape[1]):
    axes[0, t].imshow(frames[:3, t].permute(1, 2, 0).numpy())
    axes[0, t].set_title(f"frame {t}, {poses[t, :3].norm():.2f} m in")
    axes[1, t].imshow(frames[3, t].numpy(), cmap="viridis", vmin=0, vmax=1)
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()

## Lifting into a canonical volume

Pixels become rays, depth places them, pose puts every frame in one shared box. Panel 3 is a viewpoint no camera occupied.

In [ ]:
# 1. what one camera saw   2. the volume it built   3. rendered back from a pose it never saw
voxel = float(((bounds[1] - bounds[0]) / torch.tensor(grid_res)).min())
samples = int(4 * 4.0 / voxel)  # what render_view derives: four per smallest voxel
print(f"{voxel*100:.1f} cm voxels -> {samples} ray samples")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

axes[0].imshow(frames[:3, 0].permute(1, 2, 0).numpy())
axes[0].set_title("1. frame 0, one 2D view")

occupancy = volume[3]
axes[1].imshow(occupancy.sum(1).T.numpy(), origin="lower", cmap="magma", aspect="auto")
axes[1].set_title("2. the volume, seen from above")
axes[1].set_xlabel("x, metres across"); axes[1].set_ylabel("z, metres away")

novel = poses[4].clone()
novel[0] -= 0.6  # step left of anywhere the camera actually stood
axes[2].imshow(render_view(volume, novel, bounds, render_hw, 90.0).permute(1, 2, 0).clamp(0, 1).numpy())
axes[2].set_title(f"3. novel pose, {samples} samples")

for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()

## Why 32x24x32

Finer voxels do not subdivide the surface, they perforate it.

In [ ]:
# why 32x24x32: finer voxels do not subdivide the surface, they perforate it
points = frames.shape[1] * frames.shape[2] * frames.shape[3]
for res in [(32, 24, 32), (64, 48, 64), (96, 72, 96)]:
    occupied = int((lift_window(frames, poses, bounds, res, 90.0)[3] > 0).sum())
    voxel_cm = float(((bounds[1] - bounds[0]) / torch.tensor(res)).min()) * 100
    print(f"{str(res):>14}  {voxel_cm:4.1f} cm voxels  {occupied / (res[0]*res[1]*res[2]) * 100:5.2f}% occupied"
          f"  {points / occupied:6.1f} points per occupied voxel")

## Hard binning against splatting

Splatting spreads each point over the eight voxels around it.

In [ ]:
# hard binning drops each point in one voxel; splatting spreads it over the eight around it
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, splatting in zip(axes, (False, True)):
    vol = lift_window(frames, poses, bounds, grid_res, 90.0, include_splatting=splatting)
    ax.imshow(render_view(vol, novel, bounds, render_hw, 90.0).permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(f"{'splat' if splatting else 'hard bin'}, "
                 f"{(vol[3] > 1e-6).float().mean() * 100:.1f}% occupied")

hard = lift_window(frames, poses, bounds, grid_res, 90.0, include_splatting=False)
soft = lift_window(frames, poses, bounds, grid_res, 90.0)
difference = (render_view(soft, novel, bounds, render_hw, 90.0)
              - render_view(hard, novel, bounds, render_hw, 90.0)).abs().mean(0)
axes[2].imshow(difference.numpy(), cmap="inferno")
axes[2].set_title("what splatting filled in")

for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()

## Scrubbing novel views

Pre-rendered sweep; parallax should be visible as you drag.

In [ ]:
# pre-render a sweep, then scrub through it -- 180 ms a frame is too slow to render live
from ipywidgets import IntSlider, interact

sweep_hw = (320, 320)
offsets = torch.linspace(-1.0, 1.0, 21)
sweep = []
for dx in tqdm(offsets, desc="rendering sweep"):
    pose = poses[2].clone()
    pose[0] += dx
    sweep.append(render_view(volume, pose, bounds, sweep_hw, 90.0).permute(1, 2, 0).clamp(0, 1).numpy())

fig, ax = plt.subplots(figsize=(6, 6))
shown = ax.imshow(sweep[len(sweep) // 2])
ax.set_xticks([]); ax.set_yticks([])

@interact(i=IntSlider(min=0, max=len(sweep) - 1, value=len(sweep) // 2, description="view"))
def scrub(i):
    shown.set_data(sweep[i])
    ax.set_title(f"camera {offsets[i]:+.2f} m sideways from frame 2")
    fig.canvas.draw_idle()